<a href="https://colab.research.google.com/github/asema405/Maachine_Learning/blob/main/ML_PR7_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

sns.set_theme(style="whitegrid")

print("Китепканалар ийгиликтүү жүктөлдү!")

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

df = pd.read_csv(url)

print("Датасеттин өлчөмү:", df.shape)
display(df.head())

In [ ]:
print("Мамычалар:")
print(df.columns.tolist())

print("\nМаалымат түрлөрү:")
df.info()

print("\nБош маанилер:")
print(df.isnull().sum())

print("\nСтатистикалык маалымат:")
display(df.describe())

In [ ]:
target = "tip"

X = df.drop(columns=[target])
y = df[target]

numeric_features = ["total_bill", "size"]
categorical_features = ["sex", "smoker", "day", "time"]

print("X өлчөмү:", X.shape)
print("y өлчөмү:", y.shape)

In [ ]:
# 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Алдын ала иштетүү пайплайны даяр!")

In [ ]:
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

rf_default = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

tree_model.fit(X_train, y_train)
rf_default.fit(X_train, y_train)

print("Decision Tree окутулду!")
print("Random Forest окутулду!")

In [ ]:
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

rf_default = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

tree_model.fit(X_train, y_train)
rf_default.fit(X_train, y_train)

print("Decision Tree окутулду!")
print("Random Forest окутулду!")

In [ ]:
comparison = pd.DataFrame([
    evaluate_regression(
        tree_model, X_test, y_test, "Decision Tree"
    ),
    evaluate_regression(
        rf_default, X_test, y_test, "Random Forest (100)"
    )
])

display(comparison.round(3))

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Эң жакшы параметрлер:")
print(grid_search.best_params_)

print("\nКросс-валидациядагы эң жакшы RMSE:")
print(-grid_search.best_score_)

In [ ]:
rf_tuned = grid_search.best_estimator_

tuned_comparison = pd.DataFrame([
    evaluate_regression(
        rf_default, X_test, y_test, "Random Forest — баштапкы"
    ),
    evaluate_regression(
        rf_tuned, X_test, y_test, "Random Forest — tuned"
    )
])

display(tuned_comparison.round(3))

In [ ]:
perm = permutation_importance(
    rf_tuned,
    X_val,
    y_val,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Признак": X_val.columns,
    "Важность": perm.importances_mean,
    "Стандартное отклонение": perm.importances_std
}).sort_values("Важность", ascending=False)

display(importance_df.round(4))

In [ ]:
plt.figure(figsize=(9, 5))

plt.barh(
    importance_df["Признак"],
    importance_df["Важность"],
    xerr=importance_df["Стандартное отклонение"]
)

plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Permutation importance (neg RMSE)")
plt.ylabel("Баштапкы белги")
plt.title("Random Forest — белгилердин маанилүүлүгү")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
least_important_feature = importance_df.iloc[-1]["Признак"]

print("Эң аз маанилүү белги:", least_important_feature)
print("Бардык белгилер:", list(X.columns))

In [ ]:
# Train жана validation маалыматтарын бириктиребиз.
# Модель тандоо test маалыматтарын карабастан бүттү.
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

reduced_features = [
    col for col in X.columns
    if col != least_important_feature
]

reduced_numeric = [
    col for col in numeric_features
    if col in reduced_features
]

reduced_categorical = [
    col for col in categorical_features
    if col in reduced_features
]

reduced_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]), reduced_numeric),

    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), reduced_categorical)
])

best_params = grid_search.best_params_

reduced_model = Pipeline(steps=[
    ("preprocessor", reduced_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=best_params["model__n_estimators"],
        max_depth=best_params["model__max_depth"],
        min_samples_leaf=best_params["model__min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    ))
])

reduced_model.fit(
    X_trainval[reduced_features],
    y_trainval
)

print("Белгиси кыскартылган модель окутулду!")
print("Калган белгилер:", reduced_features)

In [ ]:
# Толук tuned модельди train + validation маалыматында кайра окутуу
final_full_model = grid_search.best_estimator_

final_full_model.fit(X_trainval, y_trainval)

full_result = evaluate_regression(
    final_full_model, X_test, y_test,
    "Бардык белгилер"
)

reduced_result = evaluate_regression(
    reduced_model, X_test[reduced_features], y_test,
    "Эң аз маанилүү белги алынган"
)

final_comparison = pd.DataFrame([
    full_result,
    reduced_result
])

display(final_comparison.round(3))

In [ ]:
all_results = pd.DataFrame([
    evaluate_regression(
        tree_model, X_test, y_test, "Decision Tree"
    ),
    evaluate_regression(
        rf_default, X_test, y_test, "Random Forest (100)"
    ),
    evaluate_regression(
        final_full_model, X_test, y_test, "Random Forest tuned"
    ),
    evaluate_regression(
        reduced_model, X_test[reduced_features], y_test,
        "Random Forest reduced"
    )
])

display(all_results.round(3))

all_results.set_index("Модель")[["RMSE", "MAE"]].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Моделдердин каталарын салыштыруу")
plt.ylabel("Ката мааниси — төмөн болгону жакшы")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# Практическая работа № 7. Случайный лес

## 1. Цель работы

Обучить ансамблевую модель случайного леса, сравнить её с одиночным деревом решений, подобрать гиперпараметры с помощью кросс-валидации и исследовать важность признаков.

## 2. Данные

В работе использован датасет `tips.csv`. Целевая переменная — `tip`, размер чаевых. Для предсказания использованы признаки `total_bill`, `size`, `sex`, `smoker`, `day` и `time`.

Данные разделены на обучающую, валидационную и тестовую выборки. Для обработки категориальных признаков применён One-Hot Encoding.

## 3. Сравнение моделей

Выполнено сравнение `DecisionTreeRegressor` и `RandomForestRegressor` с 100 деревьями.

| Модель | MAE | RMSE | R² |
|---|---:|---:|---:|
| Decision Tree | [значение] | [значение] | [значение] |
| Random Forest (100) | [значение] | [значение] | [значение] |
| Random Forest tuned | [значение] | [значение] | [значение] |

Для сравнения использовались одинаковые данные. MAE и RMSE характеризуют величину ошибки предсказания, а R² — качество объяснения вариации целевой переменной.

## 4. Настройка гиперпараметров

С помощью `GridSearchCV` проведён перебор параметров `n_estimators`, `max_depth` и `min_samples_leaf` с 5-блочной кросс-валидацией.

Лучшие параметры:

- `n_estimators`: [значение];
- `max_depth`: [значение];
- `min_samples_leaf`: [значение].

Количество деревьев выбиралось в диапазоне 100 и 200. Увеличение количества деревьев позволяет усреднять предсказания большего числа моделей, однако повышает время обучения. Поэтому 100 деревьев использованы как начальная точка, а 200 проверены при настройке.

## 5. Важность признаков

Для анализа использована permutation importance на валидационной выборке. Этот метод оценивает, насколько ухудшается качество модели после случайного перемешивания значений отдельного признака.

Наиболее важные признаки по результатам эксперимента:

1. [название признака];
2. [название признака].

Наименее важный признак: [].

Важность признака отражает его полезность для предсказаний конкретной модели на данном наборе данных и не доказывает причинно-следственную связь.

## 6. Эксперимент с удалением признака

Наименее важный признак удалён, после чего модель обучена повторно. Итоговые результаты на тестовой выборке:

- RMSE с полным набором признаков: [значение];
- RMSE после удаления признака: [значение];
- R² с полным набором признаков: [значение];
- R² после удаления признака: [значение].

Если качество после удаления признака почти не изменилось, это указывает на возможность упростить набор признаков без заметной потери качества в данном эксперименте. Если качество ухудшилось, признак может содержать полезную информацию.

## 7. Вывод

В ходе работы были обучены дерево решений и случайный лес, выполнена настройка гиперпараметров с помощью кросс-валидации и проведён анализ важности признаков.

Сравнение показателей MAE, RMSE и R² позволило оценить качество моделей. Эксперимент с удалением наименее важного признака показал, насколько сокращение входных данных влияет на предсказания.

Окончательный выбор модели сделан на основе результатов измерений, а не предположения о том, что ансамбль всегда должен быть лучше одиночного дерева.
